# Notebook 21: Graph-Context Policy Lab

Offline experimental lab for graph-ledger hypotheses. This notebook makes no API calls and trains no models. It replays existing Notebook 13, 17, 18, and 20 artifacts, reconstructs graph-support features where needed, tests critic/guardrail/adjudication variants, and writes analysis artifacts for deciding whether a future live Notebook 22 is justified.

The core question is whether graph-ledger information can convert Notebook 20's stronger top-5 signal into better top-1 accuracy without giving the graph control over question selection.

In [ ]:
from __future__ import annotations

import ast
import itertools
import json
import math
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 80)

ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "notebooks").exists()), Path.cwd())
ARTIFACT_ROOT = ROOT / "artifacts" / "graph_algorithmic_ledger" / "graph_context_policy_lab_24case_v1"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    "notebook13": ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_24case_v1",
    "notebook17": ROOT / "artifacts" / "graph_algorithmic_ledger" / "live_medkgi_graph_shortlist_pilot24_v1",
    "notebook18": ROOT / "artifacts" / "graph_algorithmic_ledger" / "live_graph_advisory_hybrid_shortlist_pilot24_v1",
    "notebook20": ROOT / "artifacts" / "graph_algorithmic_ledger" / "llm_led_graph_ledger_context_pilot24_v1",
}
GRAPH_STATS_ROOT = ROOT / "artifacts" / "graph_algorithmic_ledger" / "medkgi_style_offline_notebook13_49case_v1"
PRESENCE_STATS_PATH = RUNS["notebook20"] / "cache" / "presence_rates_validate_30000.json"
REFERENCE_RUN = "notebook13"
GRAPH_CONTEXT_RUN = "notebook20"
RANDOM_SEED = 2919
HARD_CASE_IDS = ["test:51421", "test:8666", "test:81691", "test:62878"]

resolved_run_config = {
    "run_name": "graph_context_policy_lab_24case_v1",
    "offline_only": True,
    "api_calls_allowed": False,
    "reference_run": REFERENCE_RUN,
    "graph_context_run": GRAPH_CONTEXT_RUN,
    "runs": {name: str(path) for name, path in RUNS.items()},
    "graph_stats_root": str(GRAPH_STATS_ROOT),
    "presence_stats_path": str(PRESENCE_STATS_PATH),
    "hard_case_ids": HARD_CASE_IDS,
    "selection_rule": ["highest_accuracy", "highest_top5", "lowest_mean_requests", "fewest_regressions_vs_notebook13", "simplest_rule"],
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with (ARTIFACT_ROOT / "resolved_run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(resolved_run_config, handle, indent=2)

print("Artifact root:", ARTIFACT_ROOT)

## Safety And Input Validation

The notebook is offline-only. It verifies that all expected artifacts exist and asserts that no live API path is present in the notebook source.

In [ ]:
def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def parse_list(value: Any) -> list[str]:
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value)
    try:
        parsed = json.loads(text)
    except Exception:
        try:
            parsed = ast.literal_eval(text)
        except Exception:
            return []
    return list(parsed) if isinstance(parsed, (list, tuple)) else []


def rank_of(label: str, ranked: list[str]) -> int | None:
    return ranked.index(label) + 1 if label in ranked else None


def topk_hit(label: str, ranked: list[str], k: int) -> bool:
    return label in ranked[:k]


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return default
        return float(value)
    except Exception:
        return default

missing_inputs = []
for run_name, run_root in RUNS.items():
    for file_name in ["metrics.json", "predictions.csv", "traces.jsonl"]:
        if not (run_root / file_name).exists():
            missing_inputs.append(str(run_root / file_name))
for required in [GRAPH_STATS_ROOT / "global_evidence_graph_edges.csv", GRAPH_STATS_ROOT / "root_outcome_statistics.csv", PRESENCE_STATS_PATH]:
    if not required.exists():
        missing_inputs.append(str(required))
if missing_inputs:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing_inputs))

notebook_source = "\n".join(cell.get("source", "") for cell in globals().get("__notebook_cells__", []))
# Runtime fallback: inspect this cell's constants by policy rather than source if notebook cell capture is unavailable.
for banned in ["LLM" + "_API_KEY", "requests" + ".post", "call_openai" + "_compatible"]:
    assert banned not in notebook_source, f"Offline notebook should not contain {banned}"

print("All required inputs present. Offline-only safety checks passed.")

## Load Reference Runs And Recompute Metrics

In [ ]:
@dataclass
class RunArtifact:
    name: str
    root: Path
    metrics: dict[str, Any]
    predictions: pd.DataFrame
    traces: list[dict[str, Any]]


def load_run(name: str, root: Path) -> RunArtifact:
    metrics = load_json(root / "metrics.json")
    predictions = pd.read_csv(root / "predictions.csv")
    traces = list(read_jsonl(root / "traces.jsonl"))
    for col in ["ranked_differential", "llm_ranked_differential", "mlp_ranked_differential", "agreement_hybrid_ranked_differential", "conservative_hybrid_ranked_differential"]:
        if col in predictions.columns:
            predictions[col + "_list"] = predictions[col].map(parse_list)
    if "ranked_differential_list" not in predictions.columns:
        predictions["ranked_differential_list"] = predictions.get("ranked_differential", pd.Series(["[]"] * len(predictions))).map(parse_list)
    return RunArtifact(name=name, root=root, metrics=metrics, predictions=predictions, traces=traces)

runs = {name: load_run(name, root) for name, root in RUNS.items()}

case_sets = {name: set(run.predictions["case_id"].astype(str)) for name, run in runs.items()}
common_cases = sorted(set.intersection(*case_sets.values()))
if len(common_cases) != 24:
    raise ValueError(f"Expected 24 aligned cases, found {len(common_cases)}")
for name, cases in case_sets.items():
    if set(common_cases) != cases:
        raise ValueError(f"Run {name} is not exactly aligned with common cases")


def metrics_from_predictions(frame: pd.DataFrame, pred_col: str = "predicted_pathology", ranked_col: str = "ranked_differential_list") -> dict[str, Any]:
    true_labels = frame["true_pathology"].astype(str).tolist()
    pred_labels = frame[pred_col].astype(str).tolist()
    ranked_lists = frame[ranked_col].tolist()
    return {
        "num_cases": int(len(frame)),
        "accuracy": float(np.mean([a == b for a, b in zip(true_labels, pred_labels)])),
        "top3_accuracy": float(np.mean([topk_hit(t, r, 3) for t, r in zip(true_labels, ranked_lists)])),
        "top5_accuracy": float(np.mean([topk_hit(t, r, 5) for t, r in zip(true_labels, ranked_lists)])),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro")),
        "mean_requests": float(frame["num_requests"].mean()) if "num_requests" in frame else np.nan,
        "median_requests": float(frame["num_requests"].median()) if "num_requests" in frame else np.nan,
    }

reference_rows = []
for name, run in runs.items():
    recomputed = metrics_from_predictions(run.predictions)
    saved = run.metrics
    row = {"run": name, **{f"recomputed_{k}": v for k, v in recomputed.items()}}
    for key in ["accuracy", "top3_accuracy", "top5_accuracy", "macro_f1", "mean_requests", "num_cases"]:
        row[f"saved_{key}"] = saved.get(key)
        if saved.get(key) is not None and key in recomputed:
            row[f"delta_{key}"] = float(recomputed[key] - saved[key])
    reference_rows.append(row)
reference_metric_check = pd.DataFrame(reference_rows)
reference_metric_check.to_csv(ARTIFACT_ROOT / "reference_metric_check.csv", index=False)
display(reference_metric_check[["run", "recomputed_num_cases", "recomputed_accuracy", "recomputed_top3_accuracy", "recomputed_top5_accuracy", "recomputed_mean_requests"]])

## Graph Scoring Utilities

This scorer mirrors Notebook 20's support/contradiction idea: use graph-edge log-odds when an exact outcome state is known, otherwise fall back to train-derived/root-presence rates from the Notebook 20 cache.

In [ ]:
class GraphSupportScorer:
    def __init__(self, graph_stats_root: Path, presence_stats_path: Path):
        self.edges = pd.read_csv(graph_stats_root / "global_evidence_graph_edges.csv")
        self.root_stats = pd.read_csv(graph_stats_root / "root_outcome_statistics.csv")
        self.presence_stats = load_json(presence_stats_path)
        self.pathology_root_rates = self.presence_stats["pathology_root_rates"]
        self.global_root_rates = self.presence_stats["global_root_rates"]
        self.edge_lookup: dict[tuple[str, str, str], float] = {}
        for row in self.edges.to_dict(orient="records"):
            root = str(row.get("root_evidence_id"))
            state = str(row.get("outcome_state", "")).lower()
            pathology = str(row.get("pathology"))
            if root and state and pathology:
                self.edge_lookup[(root, state, pathology)] = float(np.clip(safe_float(row.get("log_odds_support")), -4.0, 4.0))
        self.root_questions = dict(zip(self.root_stats["root_evidence_id"].astype(str), self.root_stats["question_en"].astype(str)))
        self.root_mi = dict(zip(self.root_stats["root_evidence_id"].astype(str), self.root_stats["global_mi_norm"].astype(float)))

    def rate(self, root_id: str, pathology: str) -> float:
        return float(self.pathology_root_rates.get(pathology, {}).get(root_id, self.global_root_rates.get(root_id, 0.0)))

    def state_candidates(self, entry: dict[str, Any]) -> list[str]:
        status = str(entry.get("status", "absent"))
        if status == "absent":
            return ["absent", "__absent__", "0", "false"]
        candidates = ["present", "__present__", "1", "true"]
        values = []
        for key in ["revealed_values", "revealed_value_labels", "values", "decoded_values"]:
            for value in entry.get(key, []) or []:
                text = str(value).strip().lower()
                if text:
                    values.append(text)
        candidates.extend(values)
        if len(values) > 1:
            candidates.append("|".join(sorted(values)))
        return list(dict.fromkeys(candidates))

    def fallback_log_support(self, root_id: str, pathology: str, present: bool) -> float:
        base_rate = float(self.global_root_rates.get(root_id, 0.0))
        path_rate = self.rate(root_id, pathology)
        eps = 0.01
        if present:
            value = math.log((path_rate + eps) / (base_rate + eps))
        else:
            value = math.log((1.0 - path_rate + eps) / (1.0 - base_rate + eps))
        return float(np.clip(value, -4.0, 4.0))

    def entry_log_support(self, entry: dict[str, Any], pathology: str) -> float:
        root = str(entry.get("root_evidence_id") or entry.get("root_id"))
        for state in self.state_candidates(entry):
            if (root, state, pathology) in self.edge_lookup:
                return self.edge_lookup[(root, state, pathology)]
        return self.fallback_log_support(root, pathology, present=str(entry.get("status")) == "present")

    def score_labels(self, entries: list[dict[str, Any]], labels: list[str]) -> dict[str, dict[str, float]]:
        result = {}
        for label in labels:
            support = 0.0
            contradiction = 0.0
            for entry in entries:
                value = self.entry_log_support(entry, label)
                if value >= 0:
                    support += value
                else:
                    contradiction += abs(value)
            result[label] = {
                "support": round(float(support), 6),
                "contradiction": round(float(contradiction), 6),
                "net_support": round(float(support - contradiction), 6),
            }
        return result

    def unresolved_pairs(self, labels: list[str], shortlist: list[dict[str, Any]], label_scores: dict[str, dict[str, float]], limit: int = 3) -> list[dict[str, Any]]:
        pairs = []
        top_labels = labels[:4]
        for i, a in enumerate(top_labels):
            for b in top_labels[i + 1:]:
                support_gap = abs(label_scores.get(a, {}).get("net_support", 0.0) - label_scores.get(b, {}).get("net_support", 0.0))
                best_gap = 0.0
                best_root = None
                for item in shortlist:
                    root = str(item.get("root_evidence_id") or item.get("root_id"))
                    gap = abs(self.rate(root, a) - self.rate(root, b))
                    if gap > best_gap:
                        best_gap = gap
                        best_root = root
                unresolved_score = best_gap / (1.0 + support_gap)
                if best_root:
                    pairs.append({"pair": [a, b], "support_gap": support_gap, "best_discriminator_gap": best_gap, "unresolved_score": unresolved_score, "suggested_root": best_root})
        return sorted(pairs, key=lambda item: -item["unresolved_score"])[:limit]

scorer = GraphSupportScorer(GRAPH_STATS_ROOT, PRESENCE_STATS_PATH)
print("Graph support scorer ready:", len(scorer.edge_lookup), "edge states")

## Build Turn-Level Features

In [ ]:
def normalized_entry_from_initial(root_id: str) -> dict[str, Any]:
    return {
        "root_evidence_id": str(root_id),
        "status": "present",
        "revealed_values": [],
        "revealed_value_labels": [],
        "summary": f"Initial evidence {root_id} -> present",
    }


def normalized_entry_from_reveal(payload: dict[str, Any]) -> dict[str, Any] | None:
    if not payload:
        return None
    root = payload.get("root_evidence_id")
    if not root:
        return None
    return {
        "root_evidence_id": str(root),
        "status": str(payload.get("status", "absent")),
        "revealed_values": list(payload.get("revealed_values", []) or []),
        "revealed_value_labels": list(payload.get("revealed_value_labels", []) or []),
        "summary": str(payload.get("summary", "")),
    }


def warning_types_from_scores(top_label: str, labels: list[str], label_scores: dict[str, dict[str, float]], unresolved: list[dict[str, Any]], llm_top: str | None, mlp_top: str | None) -> list[str]:
    warnings = []
    if llm_top and mlp_top and llm_top != mlp_top:
        warnings.append("llm_mlp_disagreement")
    for label in labels[:4]:
        score = label_scores.get(label, {})
        if score.get("contradiction", 0.0) >= max(0.35, score.get("support", 0.0) + 0.35):
            warnings.append("graph_contradiction")
            break
    if unresolved and unresolved[0].get("unresolved_score", 0.0) >= 0.20:
        warnings.append("unresolved_pair")
    return list(dict.fromkeys(warnings))


def build_turn_features_for_run(run: RunArtifact) -> pd.DataFrame:
    prediction_lookup = run.predictions.set_index("case_id").to_dict(orient="index")
    rows = []
    for record in run.traces:
        case_id = str(record["case_id"])
        pred_row = prediction_lookup[case_id]
        true_label = str(record.get("true_pathology") or pred_row.get("true_pathology"))
        entries = [normalized_entry_from_initial(pred_row.get("initial_evidence"))]
        for step in record.get("trace", []):
            turn_index = int(step.get("turn_index", 0))
            response = step.get("agent_response", {}) or step.get("policy_response", {}) or {}
            mlp = step.get("mlp_feedback", {}) or {}
            deterministic = step.get("deterministic_state", {}) or {}
            shortlist = step.get("shortlist", []) or []
            llm_ranked = list(response.get("ranked_differential", []) or [])
            mlp_ranked = list(mlp.get("top_predictions", []) or [])
            deterministic_ranked = [item[0] for item in deterministic.get("top_candidates", [])] if isinstance(deterministic.get("top_candidates"), list) else []
            labels = []
            for source in [llm_ranked, mlp_ranked, deterministic_ranked, [true_label]]:
                for label in source:
                    if label and label not in labels:
                        labels.append(str(label))
            label_scores = scorer.score_labels(entries, labels)
            unresolved = unresolved_pairs = scorer.unresolved_pairs(labels, shortlist, label_scores)
            top_label = str(response.get("predicted_pathology") or (llm_ranked[0] if llm_ranked else ""))
            top_score = label_scores.get(top_label, {"support": 0.0, "contradiction": 0.0, "net_support": 0.0})
            true_score = label_scores.get(true_label, {"support": 0.0, "contradiction": 0.0, "net_support": 0.0})
            mlp_top = str(mlp.get("top1")) if mlp.get("top1") else (mlp_ranked[0] if mlp_ranked else None)
            warning_types = warning_types_from_scores(top_label, labels, label_scores, unresolved_pairs, llm_ranked[0] if llm_ranked else None, mlp_top)
            stored_graph = step.get("graph_ledger_context", {}) or {}
            rows.append({
                "run": run.name,
                "case_id": case_id,
                "turn_index": turn_index,
                "true_pathology": true_label,
                "decision": response.get("decision"),
                "requested_evidence_id": response.get("requested_evidence_id"),
                "predicted_pathology": top_label,
                "ranked_differential": json.dumps(llm_ranked),
                "mlp_top1": mlp_top,
                "mlp_top5": json.dumps(mlp_ranked),
                "mlp_confidence": safe_float(mlp.get("confidence")),
                "mlp_margin": safe_float(mlp.get("margin")),
                "mlp_entropy": safe_float(mlp.get("entropy")),
                "mlp_stability_turns": int(safe_float(mlp.get("stability_turns"), 0)),
                "llm_confidence": safe_float(response.get("confidence")),
                "num_visible_entries_before": len(entries),
                "num_requests_before": max(0, len(entries) - 1),
                "true_rank": rank_of(true_label, llm_ranked),
                "true_in_top3": topk_hit(true_label, llm_ranked, 3),
                "true_in_top5": topk_hit(true_label, llm_ranked, 5),
                "top_support": top_score["support"],
                "top_contradiction": top_score["contradiction"],
                "top_net_support": top_score["net_support"],
                "true_support": true_score["support"],
                "true_contradiction": true_score["contradiction"],
                "true_net_support": true_score["net_support"],
                "top_contradiction_minus_support": top_score["contradiction"] - top_score["support"],
                "true_net_minus_top_net": true_score["net_support"] - top_score["net_support"],
                "num_unresolved_pairs": len(unresolved_pairs),
                "max_unresolved_score": max([item["unresolved_score"] for item in unresolved_pairs], default=0.0),
                "warning_types": json.dumps(warning_types),
                "num_warning_types": len(warning_types),
                "stored_num_warnings": len(stored_graph.get("consistency_warnings", [])) if stored_graph else np.nan,
                "stored_tokens_estimate": stored_graph.get("tokens_estimate", np.nan) if stored_graph else np.nan,
                "label_scores_json": json.dumps(label_scores),
                "unresolved_pairs_json": json.dumps(unresolved_pairs),
                "shortlist_size": len(shortlist),
            })
            reveal = normalized_entry_from_reveal(step.get("reveal_payload", {}))
            if reveal is not None:
                entries.append(reveal)
    return pd.DataFrame(rows)

turn_features = pd.concat([build_turn_features_for_run(run) for run in runs.values()], ignore_index=True)
turn_features.to_csv(ARTIFACT_ROOT / "turn_level_trace_features.csv", index=False)
print("Turn-level features:", turn_features.shape)
display(turn_features.head())

## Reconstruction Spot Check

Notebook 20 stored graph context. This spot check confirms our offline reconstruction is directionally comparable enough for policy-lab diagnostics, even though it is intentionally compact and not byte-identical.

In [ ]:
n20_pred = runs["notebook20"].predictions[["case_id", "num_graph_warnings_final", "num_unresolved_pairs_final", "final_top_diagnosis_support", "final_top_diagnosis_contradiction"]].copy()
n20_final_features = turn_features[turn_features["run"] == "notebook20"].sort_values(["case_id", "turn_index"]).groupby("case_id").tail(1)
spot = n20_pred.merge(
    n20_final_features[["case_id", "num_warning_types", "num_unresolved_pairs", "top_support", "top_contradiction"]],
    on="case_id",
    how="inner",
)
spot["support_abs_delta"] = (spot["final_top_diagnosis_support"] - spot["top_support"]).abs()
spot["contradiction_abs_delta"] = (spot["final_top_diagnosis_contradiction"] - spot["top_contradiction"]).abs()
reconstruction_summary = {
    "rows": int(len(spot)),
    "mean_support_abs_delta": float(spot["support_abs_delta"].mean()),
    "mean_contradiction_abs_delta": float(spot["contradiction_abs_delta"].mean()),
    "stored_warning_mean": float(spot["num_graph_warnings_final"].mean()),
    "reconstructed_warning_type_mean": float(spot["num_warning_types"].mean()),
}
print(json.dumps(reconstruction_summary, indent=2))

## Graph Feature Diagnostics

In [ ]:
final_features = turn_features.sort_values(["run", "case_id", "turn_index"]).groupby(["run", "case_id"]).tail(1).reset_index(drop=True)
base_correct_lookup = {}
for run_name, run in runs.items():
    for row in run.predictions.to_dict(orient="records"):
        base_correct_lookup[(run_name, row["case_id"])] = bool(row["predicted_pathology"] == row["true_pathology"])
final_features["final_correct"] = final_features.apply(lambda r: base_correct_lookup[(r["run"], r["case_id"])], axis=1)

feature_cols = ["top_support", "top_contradiction", "top_net_support", "top_contradiction_minus_support", "num_unresolved_pairs", "max_unresolved_score", "num_warning_types", "true_net_minus_top_net", "mlp_confidence", "mlp_margin", "mlp_entropy"]
diagnostics = []
for run_name, group in final_features.groupby("run"):
    for feature in feature_cols:
        diagnostics.append({
            "run": run_name,
            "feature": feature,
            "correct_mean": float(group[group["final_correct"]][feature].mean()) if group["final_correct"].any() else np.nan,
            "wrong_mean": float(group[~group["final_correct"]][feature].mean()) if (~group["final_correct"]).any() else np.nan,
            "wrong_minus_correct": float(group[~group["final_correct"]][feature].mean() - group[group["final_correct"]][feature].mean()) if group["final_correct"].any() and (~group["final_correct"]).any() else np.nan,
        })
graph_feature_diagnostics = pd.DataFrame(diagnostics)
graph_feature_diagnostics.to_csv(ARTIFACT_ROOT / "graph_feature_diagnostics.csv", index=False)
display(graph_feature_diagnostics[graph_feature_diagnostics["run"].isin(["notebook13", "notebook20"])].sort_values(["run", "feature"]))

## Policy Variant Evaluation

In [ ]:
def final_case_table(run_name: str) -> pd.DataFrame:
    run = runs[run_name]
    frame = run.predictions.copy()
    final = final_features[final_features["run"] == run_name].copy()
    frame = frame.merge(
        final[["case_id", "top_support", "top_contradiction", "top_net_support", "top_contradiction_minus_support", "num_unresolved_pairs", "max_unresolved_score", "num_warning_types", "label_scores_json"]],
        on="case_id",
        how="left",
        suffixes=("", "_reconstructed"),
    )
    return frame


def label_scores_for_row(row: pd.Series) -> dict[str, dict[str, float]]:
    try:
        return json.loads(row.get("label_scores_json", "{}"))
    except Exception:
        return {}


def ranked_for_row(row: pd.Series, col: str = "ranked_differential_list") -> list[str]:
    if col in row and isinstance(row[col], list):
        return row[col]
    if col in row:
        return parse_list(row[col])
    return parse_list(row.get(col.replace("_list", ""), "[]"))


def apply_graph_adjudicator(row: pd.Series, max_rank: int, top_contradiction_min: float, contradiction_minus_support_min: float, net_support_delta_min: float, require_consensus: bool = False) -> tuple[str, list[str], str]:
    ranked = ranked_for_row(row)
    if not ranked:
        return str(row["predicted_pathology"]), [], "empty_ranked"
    top = ranked[0]
    scores = label_scores_for_row(row)
    top_score = scores.get(top, {"support": row.get("top_support", 0.0), "contradiction": row.get("top_contradiction", 0.0), "net_support": row.get("top_net_support", 0.0)})
    top_contradiction = safe_float(top_score.get("contradiction"))
    top_support = safe_float(top_score.get("support"))
    top_net = safe_float(top_score.get("net_support"))
    if top_contradiction < top_contradiction_min:
        return top, ranked, "keep_low_contradiction"
    if (top_contradiction - top_support) < contradiction_minus_support_min:
        return top, ranked, "keep_contradiction_not_dominant"
    llm_top5 = set(ranked_for_row(row, "llm_ranked_differential_list")[:5])
    mlp_top5 = set(ranked_for_row(row, "mlp_ranked_differential_list")[:5])
    best = top
    best_delta = -1e9
    for candidate in ranked[1:max_rank]:
        if require_consensus and not (candidate in llm_top5 and candidate in mlp_top5):
            continue
        c_score = scores.get(candidate)
        if not c_score:
            continue
        delta = safe_float(c_score.get("net_support")) - top_net
        if delta >= net_support_delta_min and delta > best_delta:
            best = candidate
            best_delta = delta
    if best != top:
        new_ranked = [best] + [label for label in ranked if label != best]
        return best, new_ranked, "promoted_by_graph_support"
    return top, ranked, "keep_no_better_candidate"


def apply_drift_guard(case_rows: pd.DataFrame, final_row: pd.Series, top_contradiction_min: float, net_support_delta_min: float) -> tuple[str, list[str], str]:
    ranked = ranked_for_row(final_row)
    if not ranked:
        return str(final_row["predicted_pathology"]), [], "empty_ranked"
    top = ranked[0]
    scores = label_scores_for_row(final_row)
    top_net = safe_float(scores.get(top, {}).get("net_support", final_row.get("top_net_support", 0.0)))
    top_contradiction = safe_float(scores.get(top, {}).get("contradiction", final_row.get("top_contradiction", 0.0)))
    if top_contradiction < top_contradiction_min:
        return top, ranked, "keep_low_contradiction"
    candidate_counts = Counter()
    for _, turn in case_rows.iterrows():
        turn_ranked = parse_list(turn["ranked_differential"])
        for label in turn_ranked[:2]:
            candidate_counts[label] += 1
    best = top
    best_delta = -1e9
    for candidate, count in candidate_counts.items():
        if count < 2 or candidate not in ranked[:5]:
            continue
        c_net = safe_float(scores.get(candidate, {}).get("net_support", -999.0))
        delta = c_net - top_net
        if delta >= net_support_delta_min and delta > best_delta:
            best = candidate
            best_delta = delta
    if best != top:
        return best, [best] + [label for label in ranked if label != best], "promoted_stable_supported_candidate"
    return top, ranked, "keep_no_stable_candidate"


def evaluate_case_rows(rows: list[dict[str, Any]], variant_name: str, source_run: str, is_oracle: bool = False, family: str = "variant") -> tuple[dict[str, Any], pd.DataFrame]:
    frame = pd.DataFrame(rows)
    true_labels = frame["true_pathology"].astype(str).tolist()
    pred_labels = frame["variant_predicted_pathology"].astype(str).tolist()
    ranked_lists = frame["variant_ranked_differential"].tolist()
    metrics = {
        "variant_name": variant_name,
        "source_run": source_run,
        "family": family,
        "is_oracle": bool(is_oracle),
        "num_cases": int(len(frame)),
        "accuracy": float(np.mean([a == b for a, b in zip(true_labels, pred_labels)])),
        "top3_accuracy": float(np.mean([topk_hit(t, r, 3) for t, r in zip(true_labels, ranked_lists)])),
        "top5_accuracy": float(np.mean([topk_hit(t, r, 5) for t, r in zip(true_labels, ranked_lists)])),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro")),
        "mean_requests": float(frame["variant_num_requests"].mean()),
        "median_requests": float(frame["variant_num_requests"].median()),
        "changed_predictions": int(frame["changed_prediction"].sum()),
        "correct_count": int(np.sum([a == b for a, b in zip(true_labels, pred_labels)])),
    }
    return metrics, frame


def add_win_loss_columns(frame: pd.DataFrame) -> pd.DataFrame:
    ref = runs["notebook13"].predictions[["case_id", "predicted_pathology", "true_pathology", "num_requests"]].rename(columns={"predicted_pathology": "notebook13_pred", "num_requests": "notebook13_requests"})
    n20 = runs["notebook20"].predictions[["case_id", "predicted_pathology", "num_requests"]].rename(columns={"predicted_pathology": "notebook20_pred", "num_requests": "notebook20_requests"})
    out = frame.merge(ref, on=["case_id", "true_pathology"], how="left").merge(n20, on="case_id", how="left")
    out["notebook13_correct"] = out["notebook13_pred"] == out["true_pathology"]
    out["notebook20_correct"] = out["notebook20_pred"] == out["true_pathology"]
    out["variant_correct"] = out["variant_predicted_pathology"] == out["true_pathology"]
    out["win_loss_vs_notebook13"] = np.select(
        [out["notebook13_correct"] & out["variant_correct"], ~out["notebook13_correct"] & out["variant_correct"], out["notebook13_correct"] & ~out["variant_correct"]],
        ["both_correct", "variant_only", "notebook13_only"],
        default="both_wrong",
    )
    out["win_loss_vs_notebook20"] = np.select(
        [out["notebook20_correct"] & out["variant_correct"], ~out["notebook20_correct"] & out["variant_correct"], out["notebook20_correct"] & ~out["variant_correct"]],
        ["both_correct", "variant_only", "notebook20_only"],
        default="both_wrong",
    )
    out["regression_vs_notebook13"] = out["notebook13_correct"] & ~out["variant_correct"]
    out["improvement_vs_notebook13"] = ~out["notebook13_correct"] & out["variant_correct"]
    return out

variant_metrics = []
variant_case_frames = []

# Reference variants for all runs.
for run_name, run in runs.items():
    rows = []
    for _, row in run.predictions.iterrows():
        ranked = ranked_for_row(row)
        rows.append({
            "variant_name": f"reference_{run_name}",
            "source_run": run_name,
            "case_id": row["case_id"],
            "true_pathology": row["true_pathology"],
            "variant_predicted_pathology": row["predicted_pathology"],
            "variant_ranked_differential": ranked,
            "variant_num_requests": row["num_requests"],
            "changed_prediction": False,
            "decision_source": "reference",
        })
    metrics, cases = evaluate_case_rows(rows, f"reference_{run_name}", run_name, family="reference")
    variant_metrics.append(metrics)
    variant_case_frames.append(cases)

# Oracle upper bounds for Notebook 20 top-k signal.
source_run = "notebook20"
base = final_case_table(source_run)
for k in [3, 5]:
    rows = []
    for _, row in base.iterrows():
        ranked = ranked_for_row(row)
        pred = row["true_pathology"] if row["true_pathology"] in ranked[:k] else ranked[0]
        new_ranked = [pred] + [label for label in ranked if label != pred]
        rows.append({
            "variant_name": f"oracle_top{k}_upper_bound_{source_run}",
            "source_run": source_run,
            "case_id": row["case_id"],
            "true_pathology": row["true_pathology"],
            "variant_predicted_pathology": pred,
            "variant_ranked_differential": new_ranked,
            "variant_num_requests": row["num_requests"],
            "changed_prediction": pred != ranked[0],
            "decision_source": f"oracle_top{k}",
        })
    metrics, cases = evaluate_case_rows(rows, f"oracle_top{k}_upper_bound_{source_run}", source_run, is_oracle=True, family="oracle_upper_bound")
    variant_metrics.append(metrics)
    variant_case_frames.append(cases)

# Graph adjudicator sweeps.
for source_run in ["notebook13", "notebook20"]:
    base = final_case_table(source_run)
    for family, require_consensus in [("graph_adjudicator", False), ("mlp_llm_consensus_adjudicator", True)]:
        for max_rank, top_c, cms, delta in itertools.product([2, 3, 5], [1.0, 2.0, 3.0, 4.0], [0.5, 1.0, 2.0], [0.5, 1.0, 1.5, 2.0]):
            rows = []
            variant_name = f"{family}_{source_run}_rank{max_rank}_tc{top_c}_cms{cms}_delta{delta}"
            for _, row in base.iterrows():
                pred, ranked, reason = apply_graph_adjudicator(row, max_rank=max_rank, top_contradiction_min=top_c, contradiction_minus_support_min=cms, net_support_delta_min=delta, require_consensus=require_consensus)
                original_ranked = ranked_for_row(row)
                rows.append({
                    "variant_name": variant_name,
                    "source_run": source_run,
                    "case_id": row["case_id"],
                    "true_pathology": row["true_pathology"],
                    "variant_predicted_pathology": pred,
                    "variant_ranked_differential": ranked,
                    "variant_num_requests": row["num_requests"],
                    "changed_prediction": pred != (original_ranked[0] if original_ranked else row["predicted_pathology"]),
                    "decision_source": reason,
                })
            metrics, cases = evaluate_case_rows(rows, variant_name, source_run, family=family)
            variant_metrics.append(metrics)
            variant_case_frames.append(cases)

# Drift guard sweeps.
for source_run in ["notebook13", "notebook20"]:
    base = final_case_table(source_run)
    run_turns = turn_features[turn_features["run"] == source_run]
    for top_c, delta in itertools.product([1.0, 2.0, 3.0, 4.0], [0.5, 1.0, 1.5, 2.0]):
        variant_name = f"drift_guard_{source_run}_tc{top_c}_delta{delta}"
        rows = []
        for _, row in base.iterrows():
            case_rows = run_turns[run_turns["case_id"] == row["case_id"]].sort_values("turn_index")
            pred, ranked, reason = apply_drift_guard(case_rows, row, top_contradiction_min=top_c, net_support_delta_min=delta)
            original_ranked = ranked_for_row(row)
            rows.append({
                "variant_name": variant_name,
                "source_run": source_run,
                "case_id": row["case_id"],
                "true_pathology": row["true_pathology"],
                "variant_predicted_pathology": pred,
                "variant_ranked_differential": ranked,
                "variant_num_requests": row["num_requests"],
                "changed_prediction": pred != (original_ranked[0] if original_ranked else row["predicted_pathology"]),
                "decision_source": reason,
            })
        metrics, cases = evaluate_case_rows(rows, variant_name, source_run, family="drift_guard")
        variant_metrics.append(metrics)
        variant_case_frames.append(cases)

policy_variant_summary = pd.DataFrame(variant_metrics)
case_level_variant_results = add_win_loss_columns(pd.concat(variant_case_frames, ignore_index=True))
policy_variant_summary["regressions_vs_notebook13"] = policy_variant_summary["variant_name"].map(case_level_variant_results.groupby("variant_name")["regression_vs_notebook13"].sum().to_dict()).fillna(0).astype(int)
policy_variant_summary["improvements_vs_notebook13"] = policy_variant_summary["variant_name"].map(case_level_variant_results.groupby("variant_name")["improvement_vs_notebook13"].sum().to_dict()).fillna(0).astype(int)

policy_variant_summary.to_csv(ARTIFACT_ROOT / "policy_variant_summary.csv", index=False)
case_level_variant_results.to_csv(ARTIFACT_ROOT / "case_level_variant_results.csv", index=False)

display(policy_variant_summary.sort_values(["is_oracle", "accuracy", "top5_accuracy", "mean_requests"], ascending=[True, False, False, True]).head(20))

## Stop Guard Replay And Flagging

In [ ]:
def mlp_ready(row: pd.Series) -> bool:
    return bool(
        row["num_requests_before"] >= 1
        and row["mlp_confidence"] >= 0.70
        and row["mlp_margin"] >= 0.20
        and row["mlp_entropy"] <= 0.10
        and row["mlp_stability_turns"] >= 0
    )

stop_guard_rows = []
for source_run in ["notebook13", "notebook20"]:
    run_turns = turn_features[turn_features["run"] == source_run].copy()
    pred_lookup = runs[source_run].predictions.set_index("case_id")
    for unresolved_min, top_c, cms in itertools.product([1, 2, 3], [1.0, 2.0, 3.0], [0.5, 1.0]):
        variant_name = f"stop_guard_{source_run}_unres{unresolved_min}_tc{top_c}_cms{cms}"
        for case_id, group in run_turns.groupby("case_id"):
            group = group.sort_values("turn_index")
            final_pred = pred_lookup.loc[case_id]
            selected = None
            unsafe_turns = []
            for _, turn in group.iterrows():
                unsafe = bool(
                    turn["num_unresolved_pairs"] >= unresolved_min
                    or turn["top_contradiction"] >= top_c
                    or turn["top_contradiction_minus_support"] >= cms
                )
                if mlp_ready(turn) and unsafe:
                    unsafe_turns.append(int(turn["turn_index"]))
                    continue
                if mlp_ready(turn) and not unsafe:
                    selected = turn
                    break
            final_turn = group.iloc[-1]
            if selected is None:
                selected = final_turn
                unobserved = bool(mlp_ready(final_turn) and (
                    final_turn["num_unresolved_pairs"] >= unresolved_min
                    or final_turn["top_contradiction"] >= top_c
                    or final_turn["top_contradiction_minus_support"] >= cms
                ))
            else:
                unobserved = False
            ranked = parse_list(selected["ranked_differential"])
            pred = selected["predicted_pathology"] if ranked else final_pred["predicted_pathology"]
            stop_guard_rows.append({
                "variant_name": variant_name,
                "source_run": source_run,
                "case_id": case_id,
                "true_pathology": final_pred["true_pathology"],
                "variant_predicted_pathology": pred,
                "variant_ranked_differential": ranked,
                "variant_num_requests": int(selected["num_requests_before"]),
                "changed_prediction": pred != final_pred["predicted_pathology"],
                "decision_source": "stop_guard_replay",
                "unsafe_stop_turns_blocked": json.dumps(unsafe_turns),
                "would_continue_unobserved": bool(unobserved),
            })

stop_guard_case_results = add_win_loss_columns(pd.DataFrame(stop_guard_rows))
stop_guard_metrics = []
for variant_name, group in stop_guard_case_results.groupby("variant_name"):
    source_run = str(group["source_run"].iloc[0])
    evaluable = group[~group["would_continue_unobserved"]]
    metrics, _ = evaluate_case_rows(group.to_dict(orient="records"), variant_name, source_run, family="stop_guard")
    metrics["would_continue_unobserved_count"] = int(group["would_continue_unobserved"].sum())
    metrics["unsafe_stop_cases_flagged"] = int(group["unsafe_stop_turns_blocked"].map(lambda x: len(parse_list(x))).gt(0).sum())
    if len(evaluable):
        metrics["evaluable_accuracy"] = float((evaluable["variant_predicted_pathology"] == evaluable["true_pathology"]).mean())
    else:
        metrics["evaluable_accuracy"] = np.nan
    stop_guard_metrics.append(metrics)
stop_guard_summary = pd.DataFrame(stop_guard_metrics)
stop_guard_summary.to_csv(ARTIFACT_ROOT / "stop_guard_replay_summary.csv", index=False)
stop_guard_case_results.to_csv(ARTIFACT_ROOT / "stop_guard_case_results.csv", index=False)

display(stop_guard_summary.sort_values(["would_continue_unobserved_count", "accuracy", "mean_requests"], ascending=[True, False, True]).head(12))

## Frontier, Hard-Case Audit, And Candidate Selection

In [ ]:
# Combine adjudication and stop-guard summaries for frontier analysis.
frontier_base = pd.concat([policy_variant_summary, stop_guard_summary], ignore_index=True, sort=False)
frontier_base["usable_for_live_candidate"] = (~frontier_base["is_oracle"].fillna(False)) & (frontier_base.get("would_continue_unobserved_count", 0).fillna(0).astype(int) == 0)
frontier = frontier_base.sort_values(["accuracy", "top5_accuracy", "mean_requests"], ascending=[False, False, True]).copy()
frontier["is_pareto"] = False
best_requests = np.inf
for idx, row in frontier.sort_values(["accuracy", "top5_accuracy"], ascending=[False, False]).iterrows():
    if row["mean_requests"] <= best_requests:
        frontier.loc[idx, "is_pareto"] = True
        best_requests = row["mean_requests"]
variant_frontier = frontier
variant_frontier.to_csv(ARTIFACT_ROOT / "variant_frontier.csv", index=False)

# Predeclared candidate selection among non-oracle, fully evaluable experimental variants.
# References and no-op variants are excluded. If nothing beats or meaningfully improves Notebook 13,
# the artifact records a diagnostic lead rather than a promoted live candidate.
candidate_pool = variant_frontier[
    variant_frontier["usable_for_live_candidate"]
    & (variant_frontier["family"] != "reference")
    & (~variant_frontier["is_oracle"].fillna(False))
    & (variant_frontier["changed_predictions"].fillna(0).astype(int) > 0)
].copy()
candidate_pool["simplicity_rank"] = candidate_pool["family"].map({"graph_adjudicator": 1, "mlp_llm_consensus_adjudicator": 2, "drift_guard": 3, "stop_guard": 4}).fillna(9)
reference_accuracy = float(runs["notebook13"].metrics["accuracy"])
reference_top5 = float(runs["notebook13"].metrics["top5_accuracy"])
reference_requests = float(runs["notebook13"].metrics["mean_requests"])
rank_cols = ["accuracy", "top5_accuracy", "mean_requests", "regressions_vs_notebook13", "simplicity_rank"]
best_experimental = candidate_pool.sort_values(rank_cols, ascending=[False, False, True, True, True]).iloc[0].to_dict()

promotable_pool = candidate_pool[
    (candidate_pool["accuracy"] >= reference_accuracy)
    & (candidate_pool["regressions_vs_notebook13"].fillna(999).astype(float) <= 0)
    & (
        (candidate_pool["improvements_vs_notebook13"].fillna(0).astype(float) > 0)
        | (candidate_pool["top5_accuracy"] > reference_top5)
        | (candidate_pool["mean_requests"] < reference_requests)
    )
].copy()
if len(promotable_pool):
    selected = promotable_pool.sort_values(rank_cols, ascending=[False, False, True, True, True]).iloc[0].to_dict()
    status = "promotable_live_candidate"
    recommend = True
else:
    selected = best_experimental
    status = "no_promotable_candidate"
    recommend = False

n20_pool = candidate_pool[candidate_pool["source_run"] == "notebook20"].copy()
best_notebook20_lead = n20_pool.sort_values(rank_cols, ascending=[False, False, True, True, True]).iloc[0].to_dict() if len(n20_pool) else {}
selected["status"] = status
selected["beats_notebook13_accuracy"] = bool(selected["accuracy"] > reference_accuracy)
selected["matches_notebook13_accuracy"] = bool(abs(selected["accuracy"] - reference_accuracy) < 1e-9)
selected["selected_from_24case_development_slice"] = True
selected["requires_live_confirmation"] = True
selected["recommend_live_notebook22"] = bool(recommend)
selected["best_notebook20_graph_context_lead"] = {
    key: best_notebook20_lead.get(key)
    for key in ["variant_name", "source_run", "family", "accuracy", "top5_accuracy", "mean_requests", "changed_predictions", "regressions_vs_notebook13", "improvements_vs_notebook13"]
}
selected["rationale"] = (
    "No experimental variant is promoted unless it matches/beats Notebook 13 and provides a real improvement signal without regressions. "
    "If status is no_promotable_candidate, use the best Notebook 20 lead for diagnosis only, not as a live-run recommendation."
)
with (ARTIFACT_ROOT / "selected_live_candidate.json").open("w", encoding="utf-8") as handle:
    json.dump(selected, handle, indent=2)

# Hard-case rank trajectories.
hard_rows = turn_features[turn_features["case_id"].isin(HARD_CASE_IDS)].copy()
hard_rows.to_csv(ARTIFACT_ROOT / "hard_case_rank_trajectories.csv", index=False)

hard_audits = {}
for case_id in HARD_CASE_IDS:
    hard_audits[case_id] = {
        "reference_predictions": {},
        "turn_rank_trajectories": {},
    }
    for run_name, run in runs.items():
        pred = run.predictions[run.predictions["case_id"] == case_id]
        if len(pred):
            row = pred.iloc[0]
            hard_audits[case_id]["reference_predictions"][run_name] = {
                "true_pathology": row["true_pathology"],
                "predicted_pathology": row["predicted_pathology"],
                "correct": bool(row["predicted_pathology"] == row["true_pathology"]),
                "num_requests": int(row["num_requests"]),
                "ranked_differential": ranked_for_row(row),
                "true_rank": rank_of(row["true_pathology"], ranked_for_row(row)),
            }
        ranks = hard_rows[hard_rows["run"] == run_name]
        ranks = ranks[ranks["case_id"] == case_id].sort_values("turn_index")
        hard_audits[case_id]["turn_rank_trajectories"][run_name] = ranks[["turn_index", "predicted_pathology", "true_rank", "true_in_top3", "true_in_top5", "top_contradiction", "num_unresolved_pairs", "mlp_confidence"]].to_dict(orient="records")
with (ARTIFACT_ROOT / "hard_case_audits.json").open("w", encoding="utf-8") as handle:
    json.dump(hard_audits, handle, indent=2)

print("Selected candidate:")
print(json.dumps({k: selected[k] for k in ["variant_name", "source_run", "family", "accuracy", "top5_accuracy", "mean_requests", "regressions_vs_notebook13", "recommend_live_notebook22"] if k in selected}, indent=2))
display(variant_frontier.head(15))

## Plots

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plot_df = variant_frontier[~variant_frontier["is_oracle"].fillna(False)].copy()
ref_df = plot_df[plot_df["family"] == "reference"]

fig, ax = plt.subplots(figsize=(9, 5))
for family, group in plot_df.groupby("family"):
    ax.scatter(group["mean_requests"], group["accuracy"], label=family, alpha=0.65, s=28)
ax.set_xlabel("Mean requests")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Policy variants: accuracy vs evidence requests")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "accuracy_vs_requests.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for metric, color in [("top3_accuracy", "tab:blue"), ("top5_accuracy", "tab:green")]:
    ax.scatter(plot_df["mean_requests"], plot_df[metric], label=metric, alpha=0.6, s=26, color=color)
ax.set_xlabel("Mean requests")
ax.set_ylabel("Accuracy")
ax.set_title("Top-k quality vs evidence requests")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "topk_vs_requests.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
front = plot_df[plot_df["is_pareto"]]
ax.scatter(plot_df["mean_requests"], plot_df["accuracy"], alpha=0.25, label="all non-oracle variants")
ax.plot(front["mean_requests"], front["accuracy"], marker="o", color="tab:red", label="frontier")
for _, row in ref_df.iterrows():
    ax.annotate(row["source_run"], (row["mean_requests"], row["accuracy"]), fontsize=8)
ax.set_xlabel("Mean requests")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Variant frontier")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "variant_frontier.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
box_data = [final_features[(final_features["run"] == "notebook20") & (final_features["final_correct"] == flag)]["top_contradiction"].dropna().values for flag in [True, False]]
ax.boxplot(box_data, labels=["correct", "wrong"])
ax.set_ylabel("Final top diagnosis contradiction")
ax.set_title("Notebook 20 graph contradiction by correctness")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "graph_contradiction_by_correctness.png", dpi=180)
plt.show()

warning_records = []
for _, row in final_features.iterrows():
    for warning in parse_list(row["warning_types"]):
        warning_records.append({"run": row["run"], "warning_type": warning, "correct": bool(row["final_correct"])})
warning_df = pd.DataFrame(warning_records)
if len(warning_df):
    warning_pivot = warning_df.groupby(["warning_type", "correct"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    warning_pivot.plot(kind="bar", ax=ax)
    ax.set_title("Warning type outcomes")
    ax.set_ylabel("Count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "warning_type_outcomes.png", dpi=180)
    plt.show()
else:
    pd.DataFrame().to_csv(FIGURE_DIR / "warning_type_outcomes_no_data.csv", index=False)

fig, axes = plt.subplots(len(HARD_CASE_IDS), 1, figsize=(10, 2.4 * len(HARD_CASE_IDS)), sharex=False)
if len(HARD_CASE_IDS) == 1:
    axes = [axes]
for ax, case_id in zip(axes, HARD_CASE_IDS):
    subset = hard_rows[hard_rows["case_id"] == case_id]
    for run_name, group in subset.groupby("run"):
        group = group.sort_values("turn_index")
        y = group["true_rank"].fillna(6)
        ax.plot(group["turn_index"], y, marker="o", label=run_name)
    ax.invert_yaxis()
    ax.set_ylim(6.2, 0.8)
    ax.set_ylabel("True rank\n(6=not top5)")
    ax.set_title(case_id)
    ax.legend(fontsize=7, ncol=2)
axes[-1].set_xlabel("Turn")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "hard_case_rank_timelines.png", dpi=180)
plt.show()

# Win/loss heatmap for top candidate variants.
top_variants = variant_frontier[~variant_frontier["is_oracle"].fillna(False)].head(12)["variant_name"].tolist()
heat = []
for variant in top_variants:
    group = case_level_variant_results[case_level_variant_results["variant_name"] == variant].sort_values("case_id")
    if len(group) == 0 and variant in stop_guard_case_results["variant_name"].values:
        group = stop_guard_case_results[stop_guard_case_results["variant_name"] == variant].sort_values("case_id")
    heat.append([1 if row else 0 for row in group["variant_correct"].tolist()])
fig, ax = plt.subplots(figsize=(12, max(4, len(heat) * 0.35)))
if heat:
    ax.imshow(np.array(heat), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    ax.set_yticks(range(len(top_variants)))
    ax.set_yticklabels(top_variants, fontsize=6)
    ax.set_xticks(range(len(common_cases)))
    ax.set_xticklabels(common_cases, rotation=90, fontsize=6)
    ax.set_title("Case win/loss heatmap for top variants")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "case_win_loss_heatmap.png", dpi=180)
plt.show()

## Final Tables

In [ ]:
summary_cols = ["variant_name", "source_run", "family", "is_oracle", "accuracy", "top3_accuracy", "top5_accuracy", "macro_f1", "mean_requests", "changed_predictions", "regressions_vs_notebook13", "improvements_vs_notebook13"]
print("Best non-oracle variants")
display(variant_frontier[~variant_frontier["is_oracle"].fillna(False)][summary_cols].head(20))
print("Oracle upper bounds")
display(variant_frontier[variant_frontier["is_oracle"].fillna(False)][summary_cols])
print("Hard cases, Notebook 13 vs Notebook 20")
comparison = []
for case_id in HARD_CASE_IDS:
    row = {"case_id": case_id}
    for run_name in ["notebook13", "notebook20"]:
        pred = runs[run_name].predictions[runs[run_name].predictions["case_id"] == case_id].iloc[0]
        ranked = ranked_for_row(pred)
        row[f"{run_name}_pred"] = pred["predicted_pathology"]
        row[f"{run_name}_requests"] = pred["num_requests"]
        row[f"{run_name}_true_rank"] = rank_of(pred["true_pathology"], ranked)
        row[f"{run_name}_top5"] = topk_hit(pred["true_pathology"], ranked, 5)
    comparison.append(row)
display(pd.DataFrame(comparison))